In [3]:
import pandas as pd
import numpy as np
import os

In [ ]:
raw_path = "../data/raw/superstore.csv"
output_path = "../data/processed/clean_sales.csv"

In [5]:
df=pd.read_csv(raw_path, encoding="latin1")
df.head(5)

,ï»¿Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016/11/08,2016/11/11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016/11/08,2016/11/11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016/06/12,2016/06/16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015/10/11,2015/10/18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015/10/11,2015/10/18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [6]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (9994, 21)

Columns:
['ï»¿Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [7]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)
df.columns.tolist()

['ï»¿row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

In [8]:
df.isna().sum().sort_values(ascending=False)

ï»¿row_id        0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

In [9]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")

In [10]:
print(df[["order_date", "ship_date"]].dtypes)
print("\nMissing order dates:", df["order_date"].isna().sum())
print("Missing ship dates:", df["ship_date"].isna().sum())

order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object

Missing order dates: 0
Missing ship dates: 0


In [11]:
df = df.dropna(subset=[
    "order_date",
    "customer_id",
    "product_id",
    "sales",
    "profit",
    "discount"
])

In [12]:
text_cols = [
    "customer_name",
    "segment",
    "product_name",
    "category",
    "sub_category",
    "region",
    "state",
    "city"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

In [13]:
numeric_cols = ["sales", "profit", "discount", "quantity"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [14]:
df[numeric_cols].isna().sum()

sales       0
profit      0
discount    0
quantity    0
dtype: int64

In [15]:
df = df.dropna(subset=["sales", "profit", "discount", "quantity"])

In [16]:

df = df[df["sales"] > 0]
df = df[df["quantity"] > 0]

In [17]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_day"] = df["order_date"].dt.day
df["order_month_name"] = df["order_date"].dt.month_name()
df["order_day_name"] = df["order_date"].dt.day_name()

In [18]:
df["date_id"] = df["order_date"].dt.date

In [19]:
print("Cleaned shape:", df.shape)
df.head()

Cleaned shape: (9994, 27)


,ï»¿row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,sales,quantity,discount,profit,order_year,order_month,order_day,order_month_name,order_day_name,date_id
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.9600,2,0.00,41.9136,2016,11,8,November,Tuesday,2016-11-08
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.9400,3,0.00,219.5820,2016,11,8,November,Tuesday,2016-11-08
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.6200,2,0.00,6.8714,2016,6,12,June,Sunday,2016-06-12
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.5775,5,0.45,-383.0310,2015,10,11,October,Sunday,2015-10-11
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,22.3680,2,0.20,2.5164,2015,10,11,October,Sunday,2015-10-11


In [20]:
df.columns.tolist()

['ï»¿row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit',
 'order_year',
 'order_month',
 'order_day',
 'order_month_name',
 'order_day_name',
 'date_id']

In [21]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Cleaned file saved to: {output_path}")

Cleaned file saved to: ../data/processed/clean_sales.csv


In [22]:
print("Rows:", len(df))
print("Unique customers:", df["customer_id"].nunique())
print("Unique products:", df["product_id"].nunique())
print("Date range:", df["order_date"].min(), "to", df["order_date"].max())

Rows: 9994
Unique customers: 793
Unique products: 1862
Date range: 2014-01-03 00:00:00 to 2017-12-30 00:00:00
